# monoT5_CT v7 — KL-to-base anti-forgetting adaptation (the 7th, last-card attempt)

Six prior `monoT5_CT` runs (§7e) all **catastrophically forgot**: base monoT5-3B-MED (TREC21 judged-pool
0.449; TREC22 NQS-pool 0.522) -> fine-tune collapses to 0.14-0.20. h2oloo got 0.7118 on the *same* KZ data,
so it is an **execution** wall, not an impossibility. The one axis the six never attacked: an explicit
**KL-to-base penalty** - during training, penalize the fine-tuned model's first-token distribution from
drifting off the **frozen base** (`loss = task_CE + lambda*KL(base||ft)`), so the KZ signal is *added*
without overwriting the strong 0.522 base. Paired with a **gentle LoRA** (LR 1e-5, r=8, a=8 -> scaling 1.0,
vs the 1e-3 / a-r=2 that forgot) and the TREC21 **canary** for checkpoint selection.

Reuses the validated setup (KZ data, MaxP windows, monoT5-MED-scored hard negatives, canary) from the
§7e notebook. GO if the develop eval (TREC21 judged-pool) rises well above the 0.449 base without collapse
-> then spend the TREC22 one-shot. Only un-foreclosed path to *beat* 2022 SOTA (§13h).

In [ ]:
!pip install -q git+https://github.com/semajyllek/ctmatch.git
!pip install -q transformers accelerate datasets sentencepiece requests tqdm peft
!pip install -q -U torchao

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
DATA_ROOT  = '/content/drive/MyDrive/ct_data23'
KZ_ROOT    = f'{DATA_ROOT}/evaluation/kz_data'
KZ_QRELS   = f'{KZ_ROOT}/qrels-clinical_trials.txt'
KZ_TOPICS  = f'{KZ_ROOT}/topics-2014_2015-description.topics'
KZ_FIELDS  = f'{DATA_ROOT}/kz_trial_fields.jsonl'
BASE_MODEL = 'castorini/monot5-3b-med-msmarco'
OUT_DIR    = f'{DATA_ROOT}/monot5_ct'

# h2oloo recipe: 1k steps, batch 128, LR 1e-3, Adafactor.
# LoRA fine-tuning: base weights frozen → no catastrophic forgetting.
# LR=1e-3 is fine for adapter weights (they start from zero).
# MICRO=8: T5-3B d_ff=16384 → OOM at MICRO=16 without GC.
STEPS, BATCH, MICRO, LR = 1000, 128, 8, 1e-3
MAX_LEN, DOC_CHARS = 512, 1400

# LoRA: rank-16 adapters on all attention projections.
# Trainable params ~14M vs 3B full model (~0.5%); second moments ~28MB vs 24GB for Adam.
LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.05

os.makedirs(OUT_DIR, exist_ok=True)
print('config set')

In [ ]:
import json, requests, time
from ctmatch.evaluation.eval_utils import get_kz_topic2text  # same parser the evaluator uses

# KZ topics + qrels
topic2text = get_kz_topic2text(KZ_TOPICS)
judg = {}
ncts = set()
for l in open(KZ_QRELS):
    t, _, d, r = l.split(); judg.setdefault(t, {})[d] = int(r); ncts.add(d)
print(f'KZ: {len(topic2text)} topics, {sum(len(v) for v in judg.values())} judgments, {len(ncts)} trials')

# fetch trial fields (title/condition/summary/detailed/eligibility) for judged NCT IDs (cache)
have = set()
if os.path.exists(KZ_FIELDS):
    have = {json.loads(l)['nct_id'] for l in open(KZ_FIELDS)}
todo = [n for n in ncts if n not in have]
API = 'https://clinicaltrials.gov/api/v2/studies'
FLD = ','.join(['protocolSection.identificationModule.nctId','protocolSection.identificationModule.briefTitle',
  'protocolSection.identificationModule.officialTitle','protocolSection.conditionsModule.conditions',
  'protocolSection.descriptionModule.briefSummary','protocolSection.descriptionModule.detailedDescription',
  'protocolSection.eligibilityModule.eligibilityCriteria'])
from tqdm.auto import tqdm
with open(KZ_FIELDS, 'a') as f:
    for i in tqdm(range(0, len(todo), 100), desc='fetch KZ trials'):
        b = todo[i:i+100]
        try:
            r = requests.get(API, params={'filter.ids': ','.join(b), 'pageSize': len(b), 'fields': FLD}, timeout=30)
            for s in r.json().get('studies', []):
                ps = s.get('protocolSection', {})
                idm, cm, dm, em = (ps.get(k, {}) for k in ['identificationModule','conditionsModule','descriptionModule','eligibilityModule'])
                f.write(json.dumps({'nct_id': idm.get('nctId',''),
                    'title': idm.get('officialTitle') or idm.get('briefTitle',''),
                    'condition': (cm.get('conditions') or [''])[0],
                    'summary': dm.get('briefSummary',''), 'detailed': dm.get('detailedDescription',''),
                    'eligibility': em.get('eligibilityCriteria','')}) + '\n')
        except Exception as e:
            print('batch failed', e)
        time.sleep(0.1)
fields = {r['nct_id']: r for r in map(json.loads, open(KZ_FIELDS))}
print(f'trial fields for {len(fields)} trials')

In [ ]:
# ── MaxP pre-selection + monoT5_MED hard-negative scoring (h2oloo §3.3) ─────
# Two things happen here with one base-model pass:
#   1. For each (topic, trial) pair, score all eligibility windows and cache
#      the best window (MaxP) as the training document.
#   2. Cache the MaxP score — used to RANK hard negatives.
#      h2oloo selects hard negatives as trials the BASE MODEL scores highest
#      but are labeled negative (the model's own false positives). BM25 negatives
#      are lexically similar but often already scored low by the base model →
#      weak gradient → forgetting dominates. monoT5_MED negatives are targeted.

MAXP_CACHE   = f'{DATA_ROOT}/kz_maxp_selections.jsonl'
WINDOW_CHARS = 600   # ≈ 6 short eligibility sentences / ~150 tokens
STRIDE_CHARS = 300   # 50% overlap

def eligibility_windows(elig_text):
    if not elig_text:
        return ['']
    wins, pos = [], 0
    while pos < len(elig_text):
        wins.append(elig_text[pos:pos + WINDOW_CHARS])
        if pos + WINDOW_CHARS >= len(elig_text):
            break
        pos += STRIDE_CHARS
    return wins or ['']

# Check whether an existing cache has the maxp_score field (v2 format).
# If it was written by an older run (no score), delete and regenerate.
cache_stale = False
if os.path.exists(MAXP_CACHE):
    with open(MAXP_CACHE) as _f:
        _first = _f.readline()
    if _first and 'maxp_score' not in _first:
        print('Cache is old format (no maxp_score) — deleting and regenerating.')
        os.remove(MAXP_CACHE)
        cache_stale = True

if os.path.exists(MAXP_CACHE):
    maxp_windows, maxp_scores = {}, {}
    for line in open(MAXP_CACHE):
        r = json.loads(line)
        key = (r['topic_id'], r['nct_id'])
        maxp_windows[key] = r['elig_window']
        maxp_scores[key]  = r['maxp_score']
    print(f'Loaded MaxP cache: {len(maxp_windows):,} (topic, trial) pairs')
else:
    import torch
    from transformers import T5Tokenizer, T5ForConditionalGeneration
    print('Loading base model for MaxP selection + hard-negative scoring...')
    mp_tok   = T5Tokenizer.from_pretrained(BASE_MODEL)
    mp_model = T5ForConditionalGeneration.from_pretrained(
        BASE_MODEL, torch_dtype=torch.float16, device_map='auto').eval()
    MP_TRUE  = mp_tok('true',  add_special_tokens=False).input_ids[0]
    MP_FALSE = mp_tok('false', add_special_tokens=False).input_ids[0]

    def _score_windows(topic_text, win_strs, batch=16):
        inputs = [f'Query: {topic_text} Document: {w} Relevant:' for w in win_strs]
        scores = []
        for i in range(0, len(inputs), batch):
            enc = mp_tok(inputs[i:i+batch], return_tensors='pt', padding=True,
                         truncation=True, max_length=MAX_LEN).to(mp_model.device)
            dec = torch.zeros((enc['input_ids'].shape[0], 1), dtype=torch.long,
                              device=mp_model.device)
            with torch.no_grad():
                out = mp_model(**enc, decoder_input_ids=dec).logits[:, 0, :]
            lp = torch.log_softmax(out.float(), dim=-1)
            scores.extend((lp[:, MP_TRUE] - lp[:, MP_FALSE]).cpu().tolist())
        return scores

    maxp_windows, maxp_scores = {}, {}
    with open(MAXP_CACHE, 'w') as cf:
        for tid, d2r in tqdm(judg.items(), desc='MaxP + score'):
            topic_text = topic2text.get(tid, '')
            if not topic_text:
                continue
            for nct in d2r:
                r = fields.get(nct)
                if not r:
                    continue
                elig  = r.get('eligibility', '') or ''
                title = r.get('title', '') or ''
                cond  = r.get('condition', '') or ''
                wins = eligibility_windows(elig)
                win_strs = [
                    f"title: {title} condition: {cond} eligibility: {w}"[:DOC_CHARS]
                    for w in wins
                ]
                scores = _score_windows(topic_text, win_strs)
                best_idx   = scores.index(max(scores))
                best_elig  = wins[best_idx]
                best_score = float(max(scores))
                key = (tid, nct)
                maxp_windows[key] = best_elig
                maxp_scores[key]  = best_score
                cf.write(json.dumps({
                    'topic_id': tid, 'nct_id': nct,
                    'elig_window': best_elig, 'maxp_score': best_score
                }) + '\n')

    del mp_model
    torch.cuda.empty_cache()
    import gc; gc.collect()
    print(f'MaxP complete: {len(maxp_windows):,} pairs → {MAXP_CACHE}')

In [ ]:
import random
import numpy as np
from collections import Counter
random.seed(42)

def doc_str(nct, with_desc=False):
    """Full concatenated doc string — used by KZ sanity check in the gate cell."""
    r = fields.get(nct)
    if not r: return ''
    s = f"title: {r['title']} condition: {r['condition']} eligibility: {r['eligibility']}"
    if with_desc: s += f" description: {r['detailed'] or r['summary']}"
    return s[:DOC_CHARS]

def maxp_doc_str(nct, topic_id, with_desc=False):
    """MaxP-selected eligibility window for this (topic, trial) pair."""
    r = fields.get(nct)
    if not r: return ''
    title    = r.get('title', '') or ''
    cond     = r.get('condition', '') or ''
    elig_win = maxp_windows.get((topic_id, nct),
                                (r.get('eligibility', '') or '')[:WINDOW_CHARS])
    s = f"title: {title} condition: {cond} eligibility: {elig_win}"
    if with_desc:
        desc = r.get('detailed') or r.get('summary') or ''
        s += f" description: {desc[:300]}"
    return s[:DOC_CHARS]

def hard_negs_monot5(topic_id, neg_list, k=3):
    """Top-k negatives ranked by base-model MaxP score (highest = hardest = most
    misleading to the base model). This is h2oloo's monoT5_MED hard-negative
    selection: the model's own false positives, not BM25 lexical overlap.

    Falls back to score=0 for any trial missing from the maxp_scores cache
    (shouldn't happen since all judged trials were scored in cell-maxp).
    Returns up to k hard + 1 weak (random from remaining) negatives.
    """
    if not neg_list:
        return []
    scored = sorted(neg_list,
                    key=lambda nct: maxp_scores.get((topic_id, nct), 0.0),
                    reverse=True)
    hard = scored[:k]
    weak_pool = scored[k:]
    if weak_pool:
        hard.append(random.choice(weak_pool))
    return hard

# Score distribution sanity check on the cached scores.
all_scores_sample = [maxp_scores[k] for k in list(maxp_scores)[:200]]
print(f'MaxP score sample (n=200): mean={np.mean(all_scores_sample):.2f}  '
      f'min={np.min(all_scores_sample):.2f}  max={np.max(all_scores_sample):.2f}')

examples = []
for tid, d2r in judg.items():
    pt = topic2text.get(tid)
    if pt is None: continue
    pos = [d for d, r in d2r.items() if r >= 1 and d in fields]
    neg = [d for d, r in d2r.items() if r == 0 and d in fields]
    if not pos or not neg: continue
    for d in pos:
        for wd in (False, True):  # templates (2) and (4)
            examples.append((f'Query: {pt} Document: {maxp_doc_str(d, tid, wd)} Relevant:', 'true'))
        for nd in hard_negs_monot5(tid, neg, k=3):
            examples.append((f'Query: {pt} Document: {maxp_doc_str(nd, tid, random.random() < 0.5)} Relevant:', 'false'))

random.shuffle(examples)
print(f'{len(examples):,} training examples', Counter(t for _, t in examples))

# Spot-check 3 examples to verify MaxP windows are non-empty and meaningful.
print('\nSample training examples:')
for i, (inp, label) in enumerate(random.sample(examples, 3)):
    print(f'  [{label}] {inp[:200]}...')

In [ ]:
# ── Tokenize training examples ───────────────────────────────────────────────
from transformers import T5Tokenizer
from tqdm.auto import tqdm

tok_train = T5Tokenizer.from_pretrained(BASE_MODEL)

tokenized = []
for x, y in tqdm(examples, desc='tokenize', leave=False):
    xi = tok_train(x, truncation=True, max_length=MAX_LEN, return_tensors='pt')
    yi = tok_train(y, return_tensors='pt')
    tokenized.append((xi.input_ids[0], xi.attention_mask[0], yi.input_ids[0]))
print(f'{len(tokenized):,} examples pre-tokenized')

In [ ]:
# ── TREC21 canary setup ───────────────────────────────────────────────────────
# 20 pre-tokenized TREC21 pairs (10 topics × 1 pos + 1 neg). Scored every 100
# training steps to detect catastrophic forgetting. Run this cell once per
# session before cell-train — it reuses corpus data from the gate cell if
# already loaded, otherwise fetches it itself (~2 min).
import random as _cr
from ctmatch.experiments import ExperimentConfig, load_corpus, load_eval
_cr.seed(0)

if 'id2fields' not in dir():
    print('Loading TREC21 corpus for canary...')
    _cfg = ExperimentConfig(data_root=DATA_ROOT)
    _cids, _cfs = load_corpus(_cfg)
    id2fields = dict(zip(_cids, _cfs))
    _sets = load_eval(_cfg, ['trec21'])
    rel21 = _sets['trec21']['rel_dict']
    t2t21 = _sets['trec21']['topic2text']
    print('loaded')

def _can_str(df):
    t = df.get('brief_title') or df.get('official_title') or ''
    c = df.get('conditions', '') or ''
    if isinstance(c, list): c = ', '.join(str(x) for x in c if x)
    e = df.get('eligibility', '') or ''
    return f'title: {t} condition: {c} eligibility: {e}'[:DOC_CHARS]

canary_pairs = []
_tids = list(rel21.items()); _cr.shuffle(_tids)
for _tid, _d2r in _tids:
    if len(canary_pairs) >= 20: break
    _qt = t2t21.get(_tid, '')
    if not _qt: continue
    _pos = [d for d, r in _d2r.items() if r >= 2 and d in id2fields]
    _neg = [d for d, r in _d2r.items() if r == 0 and d in id2fields]
    if not _pos or not _neg: continue
    for _d, _lbl in [(_cr.choice(_pos), 'true'), (_cr.choice(_neg), 'false')]:
        _xi = tok_train(f'Query: {_qt} Document: {_can_str(id2fields[_d])} Relevant:',
                        truncation=True, max_length=MAX_LEN, return_tensors='pt')
        canary_pairs.append((_xi.input_ids[0], _xi.attention_mask[0], _lbl))

print(f'TREC21 canary ready: {len(canary_pairs)} pairs ({len(canary_pairs)//2} topics)')

In [ ]:
# v7 hyperparameters - override the section-7e config for the gentle + anti-forgetting recipe.
LR = 1e-5                                        # was 1e-3 (aggressive; drove the forgetting)
LORA_R, LORA_ALPHA, LORA_DROPOUT = 8, 8, 0.05    # scaling alpha/r = 1.0 (was 2.0)
LAMBDA_KL = 3.0                                  # KL(base||ft) weight - the anti-forget anchor; try 1 / 3 / 10
STEPS, CKPT_EVERY = 500, 50
OUT_V7 = f'{OUT_DIR}_v7'
print(f'v7: gentle LoRA (r={LORA_R} a={LORA_ALPHA} lr={LR}) + KL-to-base lambda={LAMBDA_KL}, {STEPS} steps')

In [ ]:
# KL-regularized training: frozen base (KL target) + LoRA-trainable ft. Checkpoint-select on the canary.
import torch, gc
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from transformers import T5ForConditionalGeneration, Adafactor
from peft import get_peft_model, LoraConfig, TaskType
from tqdm.auto import tqdm
device = 'cuda' if torch.cuda.is_available() else 'cpu'

base = T5ForConditionalGeneration.from_pretrained(BASE_MODEL, torch_dtype=torch.bfloat16).to(device).eval()
for p in base.parameters(): p.requires_grad_(False)
ft = T5ForConditionalGeneration.from_pretrained(BASE_MODEL, torch_dtype=torch.bfloat16).to(device)
ft = get_peft_model(ft, LoraConfig(task_type=TaskType.SEQ_2_SEQ_LM, r=LORA_R, lora_alpha=LORA_ALPHA,
                                   lora_dropout=LORA_DROPOUT, target_modules=['q', 'v']))
ft.print_trainable_parameters(); ft.train()
opt = Adafactor(ft.parameters(), lr=LR, scale_parameter=False, relative_step=False, warmup_init=False)
TRUE_ID  = tok_train('true',  add_special_tokens=False).input_ids[0]
FALSE_ID = tok_train('false', add_special_tokens=False).input_ids[0]

class DS(Dataset):
    def __init__(s, ex): s.ex = ex
    def __len__(s): return len(s.ex)
    def __getitem__(s, i): return s.ex[i]
def collate(b):
    ii = pad_sequence([x[0] for x in b], batch_first=True, padding_value=tok_train.pad_token_id)
    am = pad_sequence([x[1] for x in b], batch_first=True, padding_value=0)
    tgt = torch.tensor([int(x[2][0]) for x in b])   # first label token = true/false id
    return ii, am, tgt
dl = DataLoader(DS(tokenized), batch_size=MICRO, shuffle=True, collate_fn=collate)

@torch.no_grad()
def canary_gap(m):
    m.eval(); pos, neg = [], []
    for ii, am, lbl in canary_pairs:
        dec = torch.zeros((1, 1), dtype=torch.long, device=device)
        out = m(input_ids=ii.unsqueeze(0).to(device), attention_mask=am.unsqueeze(0).to(device),
                decoder_input_ids=dec).logits[0, 0].float()
        lp = torch.log_softmax(out, -1); (pos if lbl == 'true' else neg).append((lp[TRUE_ID] - lp[FALSE_ID]).item())
    m.train(); return sum(pos)/len(pos) - sum(neg)/len(neg)

init_gap = canary_gap(ft); print(f'init canary gap = {init_gap:.2f} (base capability)')
ACC = BATCH // MICRO; step = 0; it = iter(dl); best = (-1e9, 0)
pbar = tqdm(total=STEPS, desc='monoT5_CT v7 (KL)')
while step < STEPS:
    opt.zero_grad(); tl = kll = 0.0
    for _ in range(ACC):
        try: ii, am, tgt = next(it)
        except StopIteration: it = iter(dl); ii, am, tgt = next(it)
        ii, am, tgt = ii.to(device), am.to(device), tgt.to(device)
        dec = torch.zeros((ii.shape[0], 1), dtype=torch.long, device=device)
        with torch.no_grad():
            bl = base(input_ids=ii, attention_mask=am, decoder_input_ids=dec).logits[:, 0, :].float()
        fl = ft(input_ids=ii, attention_mask=am, decoder_input_ids=dec).logits[:, 0, :].float()
        task = torch.nn.functional.cross_entropy(fl, tgt)
        kl = torch.nn.functional.kl_div(torch.log_softmax(fl, -1), torch.softmax(bl, -1), reduction='batchmean')
        loss = (task + LAMBDA_KL * kl) / ACC
        loss.backward(); tl += task.item()/ACC; kll += kl.item()/ACC
    torch.nn.utils.clip_grad_norm_(ft.parameters(), 1.0); opt.step(); step += 1; pbar.update(1)
    if step % 20 == 0: print(f'step {step:4d}  task={tl:.3f}  kl={kll:.3f}')
    if step % CKPT_EVERY == 0:
        g = canary_gap(ft); print(f'  canary gap = {g:.2f}  ({g/init_gap*100:.0f}% of base)')
        if g > best[0]:
            best = (g, step); ft.save_pretrained(f'{OUT_V7}_best'); tok_train.save_pretrained(f'{OUT_V7}_best')
            print(f'    ^ new best canary - saved adapter (step {step})')
pbar.close()
print(f'best canary gap {best[0]:.2f} at step {best[1]}  (base init {init_gap:.2f}) -> {OUT_V7}_best')
del base, ft; gc.collect(); torch.cuda.empty_cache()

In [ ]:
# Develop eval: the v7-best adapter on the TREC21 judged pool (comparable to base 0.449). GO/NO-GO.
import torch, numpy as np
from transformers import T5Tokenizer, T5ForConditionalGeneration
from peft import PeftModel
from ctmatch.experiments import ExperimentConfig, load_corpus, load_eval, ndcg_at_k
cfg = ExperimentConfig(data_root=DATA_ROOT)
corpus_ids, corpus_fields = load_corpus(cfg); id2fields = dict(zip(corpus_ids, corpus_fields))
s21 = load_eval(cfg, ['trec21'])['trec21']; rel21 = s21['rel_dict']; t2t21 = s21['topic2text']
etok = T5Tokenizer.from_pretrained(BASE_MODEL)
ebase = T5ForConditionalGeneration.from_pretrained(BASE_MODEL, torch_dtype=torch.float16).to(device).eval()
emodel = PeftModel.from_pretrained(ebase, f'{OUT_V7}_best').eval()
TRUE = etok('true', add_special_tokens=False).input_ids[0]; FALSE = etok('false', add_special_tokens=False).input_ids[0]
def vdoc(df):
    t = df.get('brief_title') or df.get('official_title') or ''
    c = df.get('conditions', ''); c = ', '.join(str(x) for x in c if x) if isinstance(c, list) else (c or '')
    e = df.get('eligibility', '') or ''
    return f'title: {t} condition: {c} eligibility: {e}'[:DOC_CHARS]
@torch.no_grad()
def score(topic, docs, b=8):
    out = []
    for i in range(0, len(docs), b):
        ins = [f'Query: {topic} Document: {vdoc(id2fields[d])} Relevant:' for d in docs[i:i+b]]
        enc = etok(ins, return_tensors='pt', padding=True, truncation=True, max_length=512).to(device)
        dec = torch.zeros((enc['input_ids'].shape[0], 1), dtype=torch.long, device=device)
        lp = torch.log_softmax(emodel(**enc, decoder_input_ids=dec).logits[:, 0, :].float(), -1)
        out.extend((lp[:, TRUE] - lp[:, FALSE]).cpu().tolist())
    return out
ndcgs = []
for t, d2r in rel21.items():
    if t not in t2t21: continue
    docs = [d for d in d2r if d in id2fields]
    if not docs: continue
    sc = score(t2t21[t], docs); ranked = [d for _, d in sorted(zip(sc, docs), reverse=True)]
    ndcgs.append(ndcg_at_k(ranked, d2r))
r = float(np.mean(ndcgs))
print(f'\nv7 monoT5_CT - TREC21 judged-pool NDCG@10 = {r:.4f}')
print(f'   base monoT5-MED = 0.449 | 6 prior fine-tunes = 0.14-0.20 (forgot) | h2oloo monoT5_CT = 0.71')
if r > 0.60:
    print('=> STRONG GO: KZ learned without forgetting. Spend the TREC22 one-shot (score the NQS pool vs 0.575/0.6125).')
elif r > 0.49:
    print('=> partial: beats base, no collapse - try LAMBDA_KL lower (1) for more KZ signal, re-eval before the one-shot.')
elif r > 0.40:
    print('=> KL likely too strong (learned ~= base) OR mild forget - sweep LAMBDA_KL (1 / 3 / 10).')
else:
    print('=> NO-GO: forgot again despite KL. The adaptation wall holds; section 13h ceiling stands.')